In [ ]:
from fastcore.utils import *
import fastcore.all as fc, re, math, itertools, functools, numpy as np, types, typing, dataclasses, matplotlib.pyplot as plt, collections, regex
from regex import search
from collections import Counter
from collections.abc import Iterable
from SPARQLWrapper import SPARQLWrapper
from rdflib import Graph
from claudette import *
import httpx
np.set_printoptions(linewidth=150, suppress=True)
plt.rcParams['figure.dpi'] = 50

**AI Prompt**

Hi solveit! I am Chuck, and I am working with you today! I'm an experienced Python coder, experienced with semantic web technologies like OWL,RDF and SPARQL, and I enjoy learning new things. When writing code for me, please follow fast.ai coding style: avoid comments; place single line if/loop/etc constructs on the same line; use concise variable names where appropriate. Follow the style you will see in my messages, as we work on this dialogue together. Are you ready?

**AI Response**

Hi Chuck! Great to meet you. I understand you're experienced with Python and semantic web technologies, and I'll follow fast.ai style conventions in any code we discuss. I'll keep things concise but will be happy to dive deep into technical details when needed.

I'm ready to help - what would you like to explore today?

## Wikidata Ontology Structure for QA Query
The Challenge: In the realm of knowledge graphs and semantic queries, we face a fascinating impedance mismatch. Large Language Models (LLMs) have become remarkably adept at translating natural language questions into SPARQL queries, yet they often stumble when it comes to respecting the formal semantic constraints encoded in ontologies. How might we bridge this gap between intuitive understanding and formal correctness?

Goal: Use LLM to construct query to wikidata endpoint using LLM to construct query to LLM endpoint. Use Ontology-based Query Check (OBQC) that detects errors by leveraging the ontology of the knowledge graph to check if the LLM-generated SPARQL query matches the semantic of ontology and 2) LLM Repair: use the error explanations with an LLM to repair the SPARQL query. Methodology using WikiData as a service.

WikiData has a set of issues related to it's "Ontology" that present chalenges because of it's lack of formal semantics that can be translated into rules using the OBQC approach.

Resources:

Papers: [Increasing the LLM Accuracy for Question Answering: Ontologies to the Rescue!](https://arxiv.org/html/2405.11706v1) WikiData: https://www.wikidata.org/wiki/Wikidata:WikiProject_Ontology/Cleaning_Task_Force [Class Order Disorder in Wikidata and First Fixes:](https://arxiv.org/html/2411.15550v1) [Disjointness Violations in Wikidata:](https://arxiv.org/html/2410.13707v2)


Wikidata Documentation Resources:
- [Property Constraints](https://www.wikidata.org/wiki/Help:Property_constraints_portal)
- [Relations Between RDF and Wikidata](https://www.wikidata.org/wiki/Wikidata:Relation_between_properties_in_RDF_and_in_Wikidata)

Below is a conceptual deep dive into how you might “crawl” the SPARQL output (plus additional queries) to dynamically construct a local or “ephemeral” ontology about an entity—like Douglas Adams. This helps you get more context and build a richer, more interpretable structure for an LLM (or any downstream consumer).

1. The Idea: Ephemeral Ontology Construction
	1.	You run an initial SPARQL query (like a DESCRIBE or a relevant SELECT) about some entity—say, Q42 (Douglas Adams).
	2.	The raw result gives you references to:
	•	Wikipedia pages linking to Q42.
	•	Possibly some direct instance of or subclass of statements, or other related properties (like “occupation,” “place of birth,” etc.).
	3.	You see object URIs in the results. Some are Q-codes or properties (P-codes).
	4.	You follow those Q-codes/properties by issuing additional queries (or using the same DESCRIBE data if it’s included) to fetch their labels, relationships, and constraints (like subclass of chains).
	5.	You accumulate all that into a local data structure (effectively a mini knowledge graph or ephemeral ontology).
	6.	This ephemeral ontology allows your LLM to interpret the original entity in a broader context—knowing not just the direct statements, but also the types, superclasses, etc.

In essence, it’s a BFS/DFS “crawl” across Wikidata’s knowledge graph, but limited to a certain depth or scope to keep it manageable.

2. Why Crawl?

2.1 Gathering “Subclass of” and “Instance of”
	•	instance of (P31) tells you what category/class an entity belongs to (e.g., “Douglas Adams” is an instance of “human (Q5)”).
	•	subclass of (P279) indicates hierarchical relationships among classes (e.g., “human” is a subclass of “Homo sapiens,” or “writer” is a subclass of “person,” etc.).

By pulling these statements, you can build a small hierarchy tree around Q42—understanding that “Douglas Adams” is a “human,” which is a “primate,” which is a “mammal,” etc. Or that “Douglas Adams” is an “English writer,” which is a subclass of “writer.”

2.2 Adding Labels, Descriptions, and Aliases
	•	Querying for rdfs:label or schema:description (or skos:altLabel) helps you build a dictionary.
	•	For example, Q5 might have the label “human,” Q42 has “Douglas Adams,” etc.
	•	This dictionary ensures you can present or interpret relationships in plain English (or your desired language).

2.3 Enhanced Interpretability

A local mini-ontology might eventually say:
	•	Q42 → instance of → Q5 (“human”)
	•	Q42 → occupation → Q36180 (“writer”)
	•	Q36180 → subclass of → Q215627 (“person”)
	•	Q5 → subclass of → Q10876 (“Hominidae”)
	•	…

You see how you get a chain of “human → hominid → primate,” etc.
Your LLM can then produce a statement like “Douglas Adams was an English writer, which is a type of person/human being.”

3. A Possible Crawling Algorithm

Let’s outline a breadth-first approach with constraints so we don’t go infinite:
	1.	Start with the subject of interest, e.g., Q42 (Douglas Adams).
	2.	Issue a SPARQL query (like DESCRIBE wd:Q42) or a custom SELECT that fetches direct statements (P31, P569, etc.).
	3.	Store the results in a local graph structure:
	•	Something like an adjacency list: Q42 -> [ (P31, Q5), (P21, Q6581097), ... ]
	•	Also store any literal data (date of birth, textual descriptions, etc.).
	4.	Queue: For each discovered object that’s a Q-code (e.g., Q5, Q6581097), add it to a queue if you haven’t visited it yet.
	5.	Expand to a limited depth:
	•	For each Q-code in the queue, you do a smaller query to get:
	•	rdfs:label, instance of, subclass of, or other properties you care about.
	•	Store those results, label them in your dictionary.
	6.	Stop or limit the crawl:
	•	If you go too deep, you’ll pull in vast swaths of Wikidata.
	•	Typically, you might limit to 1 or 2 levels beyond the original node or certain property types (like only expand P31, P279, P361, etc.).

Result: A partial subgraph capturing the key hierarchical and definitional relationships for the target entity.

4. Example: Douglas Adams Crawl
	1.	Initial Query:

DESCRIBE wd:Q42

or a custom SELECT that collects (subject, predicate, object) for Q42.

	2.	Results:
	•	Some data about Q42 referencing P31 = Q5 (human), P21 = Q6581097 (male), P569 = literal(“1952-03-11”), etc.
	•	Some references to Wikipedia pages.
	3.	Local Graph:
	•	Q42 -> (P31, Q5) // instance of human
	•	Q42 -> (P21, Q6581097) // sex or gender: male
	•	…
	4.	Next Step:
	•	For Q5 (human), do a query:

SELECT ?p ?o WHERE { wd:Q5 ?p ?o }

to see if it has subclass of statements.

	•	For Q6581097 (male), maybe we check if it’s a subclass of “gender” or something else.

	5.	Limited Depth:
	•	We gather 1 or 2 levels of “subclass of” for Q5 → Q215627 (person) → etc.
	6.	Labels:
	•	We do rdfs:label lookups for Q5 (“human”), Q6581097 (“male”), Q215627 (“person”), etc.
	7.	Merged:
	•	Our ephemeral ontology might look like:

Q42 (Douglas Adams)
   - instance of -> Q5 (human)
   - sex or gender -> Q6581097 (male)
   - occupation -> Q36180 (writer)
Q5 (human)
   - subclass of -> Q215627 (person)
Q36180 (writer)
   - subclass of -> Q215627 (person)
Q215627 (person)
   - ??? 
...


	8.	Interpretation:
	•	The LLM can now read from this local structure: “Douglas Adams is an instance of human, which is a subclass of person, and has occupation writer (also a subclass of person?), so he is a male human, an English writer,” etc.

5. The Role of the LLM in This Process
	1.	Parsing and Summarizing: The LLM can read your ephemeral graph and produce a human-readable summary.
	2.	Constraint Checking: If you’re doing OBQC, you can check if certain domain-range constraints are violated (like “date of birth” must be for a person, not for a corporation”).
	3.	Guided Crawling: The LLM might be used to decide which properties are worth crawling deeper, or to confirm which relationships are relevant to the user’s question.

6. Potential Pitfalls & Mitigations
	1.	Explosion of Data: Crawling too far can produce a huge subgraph.
	•	Mitigation: Limit the depth or the property types you explore (only P31, P279, P361, etc.).
	2.	Data Gaps: Some Q-items may not have subclass of or labels in the language you want.
	•	Mitigation: Use fallback languages or store partial data.
	3.	Conflicting Statements: Wikidata can have contradictory statements or multiple values.
	•	Mitigation: Let your ephemeral graph store all statements, or filter out deprecated statements.
	4.	Performance: Many repeated lookups could slow you down or hit SPARQL rate limits.
	•	Mitigation: Cache results locally, or run bulk queries that gather all needed data in one shot.

7. Why This Helps an LLM
	•	Instead of giving the LLM random “subject, predicate, object” lines, you’re providing a curated, structured subgraph.
	•	The LLM sees short labels or definitions for each Q-code or property.
	•	The LLM can more easily produce a coherent narrative about the entity, referencing its place in a knowledge hierarchy, its base attributes, etc.

8. Final Thoughts

Yes, you can absolutely dynamically “crawl” the output of a SPARQL query to build up a mini-ontology for an entity. By systematically retrieving instance of, subclass of, or other relevant relationships (plus labels), you form a local knowledge structure that’s both:
	•	Rich enough to let an LLM (or any system) interpret the entity with more context.
	•	Limited enough to keep the process feasible—so you don’t replicate the entirety of Wikidata in one go.

This ephemeral ontology, combined with typical LLM steps (like summarization and QA), forms a powerful approach for interpretability and user-friendly knowledge exploration around any given Wikidata entity.

**AI Prompt**

Summarize what we are trying to accomplish in this SolveIt session.

**AI Response**

We're working on a sophisticated knowledge graph exploration project with several key objectives:
1. Building a natural language QA system based on LLMs to SPARQL query
   - Translating natural language questions into SPARQL queries
   - Ensuring semantic correctness through Ontology-based Query Checking (OBQC)
   - Repairing queries when semantic errors are detected
   - Handling Wikidata's unique ontological challenges
   - Uses Agentic patterns for LLMs implimented using Claudette
     
2. Building an "ephemeral ontology" system that:
   - Starts with a Wikidata entity (like Douglas Adams/Q42)
   - Intelligently crawls related entities and properties
   - Constructs a focused, local knowledge structure
   - Maintains semantic meaning while keeping the scope manageable

3. Bridging the gap between LLMs and formal ontologies by:
   - Using LLMs to generate SPARQL queries
   - Implementing Ontology-based Query Checking (OBQC)
   - Providing query repair capabilities when semantic errors are detected
   - Handling Wikidata's specific ontological challenges

4. Creating a system that can:
   - Navigate through instance/subclass relationships
   - Gather relevant labels and descriptions
   - Build meaningful hierarchical structures
   - Support LLM interpretation of the knowledge context

The ultimate goal is to create a more robust and semantically accurate question-answering system that leverages both the strengths of LLMs and the formal structure of knowledge graphs.

Would you like to start working on implementing any particular aspect of this system?

**AI Prompt**

If we are using the Polya solveit methodology, where would you suggest we start to understand such a system?

**AI Response**

Following Polya's problem-solving methodology, let's break this down:

1. **Understand the Problem**
   - Let's start with a simple, concrete example to understand the flow
   - We could take Douglas Adams (Q42) as our test case and:
     * Write a basic SPARQL query manually
     * Examine its structure and results
     * Identify what makes a query "semantically correct"
     * See how the ontology constraints apply

2. **Devise a Plan**
   - After understanding the basic flow, we can plan how to:
     * Build the ephemeral ontology crawler
     * Implement the OBQC checking
     * Integrate the LLM components

3. **Carry Out the Plan**
   - Implement components incrementally
   - Test each piece thoroughly
   - Integrate gradually

4. **Look Back**
   - Review results
   - Refine approach
   - Optimize performance

I suggest we start with step 1: Write a simple SPARQL query about Douglas Adams and examine its structure and results. This will give us concrete data to work with and help us understand the challenges we'll face.

Would you like to begin with this approach?

**AI Prompt**

Where would you suggest starting?

**AI Prompt**

Ok. Lets use SPARQLWrapper since that will help us to structure the graph.

**AI Response**

I'll create a helper function to execute and format our SPARQL queries, then run our initial query about Douglas Adams:

```python
def query_wikidata(query):
    endpoint = SPARQLWrapper("https://query.wikidata.org/sparql")
    endpoint.setReturnFormat('json')
    endpoint.setQuery(query)
    try:
        results = endpoint.query().convert()
        return results['results']['bindings']
    except Exception as e: print(f"Query failed: {e}")

# More focused query to get key information about Douglas Adams
query = """
SELECT DISTINCT ?prop ?propLabel ?value ?valueLabel WHERE {
  wd:Q42 ?prop ?value .
  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "en" .
    ?prop rdfs:label ?propLabel .
    ?value rdfs:label ?valueLabel .
  }
  FILTER(STRSTARTS(STR(?prop), "http://www.wikidata.org/prop/direct/"))
} LIMIT 10
"""

results = query_wikidata(query)
if results: 
    for r in results: print(f"{r['propLabel']['value']}: {r['valueLabel']['value']}")
```

This should give us a first look at some key properties of Douglas Adams. After we see these results, we can focus on specific properties that will be most relevant for building our ephemeral ontology, particularly looking at instance-of and subclass-of relationships. Shall we run this?

In [ ]:
def query_wikidata(query):
    endpoint = SPARQLWrapper("https://query.wikidata.org/sparql")
    endpoint.setReturnFormat('json')
    endpoint.setQuery(query)
    try:
        results = endpoint.query().convert()
        return results['results']['bindings']
    except Exception as e: print(f"Query failed: {e}")

# More focused query to get key information about Douglas Adams
query = """
SELECT DISTINCT ?prop ?propLabel ?value ?valueLabel WHERE {
  wd:Q42 ?prop ?value .
  SERVICE wikibase:label { 
    bd:serviceParam wikibase:language "en" .
    ?prop rdfs:label ?propLabel .
    ?value rdfs:label ?valueLabel .
  }
  FILTER(STRSTARTS(STR(?prop), "http://www.wikidata.org/prop/direct/"))
} LIMIT 10
"""

results = query_wikidata(query)
if results: 
    for r in results: print(f"{r['propLabel']['value']}: {r['valueLabel']['value']}")

http://www.wikidata.org/prop/direct/P3553: 29259153
http://www.wikidata.org/prop/direct/P5562: adams_douglas_noel
http://www.wikidata.org/prop/direct/P2607: 307812da-da11-4ee5-a906-31e5ce9694bb
http://www.wikidata.org/prop/direct/P2604: 246164
http://www.wikidata.org/prop/direct/P2605: 39534
http://www.wikidata.org/prop/direct/P2600: 6000000050650155828
http://www.wikidata.org/prop/direct/P2611: douglas_adams
http://www.wikidata.org/prop/direct/P2626: 159696
http://www.wikidata.org/prop/direct/P2638: people/douglas-adams
http://www.wikidata.org/prop/direct/P2722: 4468812


**AI Prompt**

Ok. There is another API that is more natuarl language LLM friendly. For example:


```
import requests
 
def fetch_wikidata(params):
    url = 'https://www.wikidata.org/w/api.php'
    try:
        return requests.get(url, params=params)
    except:

 


        return 'There was and error'
```

Can we do somethink like this using httpx instead?

**AI Response**

Yes, let's create a more LLM-friendly API using httpx, which is already imported and is more modern than requests. Here's a version that will help us interact with Wikidata's API:

```python
def fetch_wikidata(params):
    url = 'https://www.wikidata.org/w/api.php'
    default_params = {
        'format': 'json',
        'action': 'wbgetentities',
        'languages': 'en'
    }
    params = {**default_params, **params}
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test it with Douglas Adams (Q42)
result = fetch_wikidata({'ids': 'Q42'})
if isinstance(result, dict): print(f"Success: Got data for {result.get('entities',{}).get('Q42',{}).get('labels',{}).get('en',{}).get('value')}")
```

This gives us a foundation to:
1. Make API calls more suited for LLM interaction
2. Get entity data in a more structured format
3. Build our ephemeral ontology more easily

Would you like to see what we get back and then expand the functionality?

Wikidata API Reference: https://www.jcchouinard.com/wikidata-api-python/

In [ ]:
def fetch_wikidata(params):
    url = 'https://www.wikidata.org/w/api.php'
    default_params = {
        'format': 'json',
        'action': 'wbgetentities',
        'languages': 'en'
    }
    params = {**default_params, **params}
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test it with Douglas Adams (Q42)
result = fetch_wikidata({'ids': 'Q42'})
if isinstance(result, dict): print(f"Success: Got data for {result.get('entities',{}).get('Q42',{}).get('labels',{}).get('en',{}).get('value')}")

Success: Got data for None


**AI Prompt**

This function returned a result none. I don't think this is what you intended. So, let's think about this. We want to make this LLM token friendly since LLM hallucinate QIDs and PIDs.

 We want to pass strings as much as possible.

**AI Response**

You're right. Let's make it more LLM-friendly by focusing on natural language strings and creating a function that can search Wikidata using text. Here's a better approach:

```python
def search_wikidata(search_text):
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': 'item'
    }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test with natural language
result = search_wikidata("Douglas Adams")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:  # Show top 3 matches
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")
```

This should:
1. Accept natural language input
2. Return human-readable results with QIDs
3. Include descriptions to help verify we found the right entity

This makes it easier for an LLM to work with the results and verify it's getting the correct entity before proceeding with more detailed queries. Shall we try this?

In [ ]:
def search_wikidata(search_text):
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': 'item'
    }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test with natural language
result = search_wikidata("Douglas Adams")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:  # Show top 3 matches
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")

ID: Q28421831, Label: Douglas Adams, Description: American environmental engineer
ID: Q42, Label: Douglas Adams, Description: English science fiction writer and humorist (1952–2001)
ID: Q61853920, Label: Douglas H Adams, Description: researcher ORCID ID = 0000-0002-3539-6629


**AI Prompt**

Ok. I think we are on the right track here since the LLM gets more structured tokens. Can this also be used to fetch informaion on properties?

**AI Response**

Yes, we can modify our function to search for properties as well. Here's an enhanced version that can search for both entities and properties:

```python
def search_wikidata(search_text, search_type='item'):
    """Search Wikidata for entities or properties
    search_type: 'item' or 'property'"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': search_type
    }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test with a property search
print("Searching for 'instance of' property:")
result = search_wikidata("instance of", "property")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")

print("\nSearching for 'occupation' property:")
result = search_wikidata("occupation", "property")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")
```

This will help us:
1. Find the correct property IDs (PIDs) using natural language
2. Get descriptions of what properties mean
3. Build more accurate queries by matching properties to their intended use

Should we run this and see what properties we get?

In [ ]:
def search_wikidata(search_text, search_type='item'):
    """Search Wikidata for entities or properties
    search_type: 'item' or 'property'"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': search_type
    }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test with a property search
print("Searching for 'instance of' property:")
result = search_wikidata("instance of", "property")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")

print("\nSearching for 'occupation' property:")
result = search_wikidata("occupation", "property")
if isinstance(result, dict):
    matches = result.get('search', [])
    if matches:
        for m in matches[:3]:
            print(f"ID: {m.get('id')}, Label: {m.get('label')}, Description: {m.get('description')}")

Searching for 'instance of' property:
ID: P31, Label: instance of, Description: that class of which this subject is a particular example and member; different from P279 (subclass of); for example: K2 is an instance of mountain; volcano is a subclass of mountain (and an instance of volcanic landform)
ID: P10241, Label: individual of taxon, Description: the taxon of an individual named organism (animal, plant)
ID: P1647, Label: subproperty of, Description: all resources related by this property are also related by that property

Searching for 'occupation' property:
ID: P106, Label: occupation, Description: occupation of a person; see also "field of work" (Property:P101), "position held" (Property:P39)
ID: P3095, Label: practiced by, Description: type of agents that study this subject or work in this field
ID: P3989, Label: members have occupation, Description: some or all members of this group share the occupation


**AI Prompt**

Ok. I think we will need tools that go the other direction. Given a PID or QID find the label and description. Does that work with these functions?

**AI Response**

Yes, we can modify our function to look up entities or properties directly by their IDs. Here's an enhanced version that can handle both directions:

```python
def lookup_wikidata(id_or_text, search_type='item', is_id=False):
    """
    Search or lookup Wikidata entities/properties
    id_or_text: QID/PID (if is_id=True) or search text
    search_type: 'item' or 'property'
    is_id: True if looking up by ID, False to search by text
    """
    url = 'https://www.wikidata.org/w/api.php'
    
    if is_id:
        params = {
            'action': 'wbgetentities',
            'format': 'json',
            'languages': 'en',
            'ids': id_or_text
        }
    else:
        params = {
            'action': 'wbsearchentities',
            'format': 'json',
            'language': 'en',
            'search': id_or_text,
            'type': search_type
        }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test direct ID lookups
print("Looking up Q42 (Douglas Adams):")
result = lookup_wikidata('Q42', is_id=True)
if isinstance(result, dict):
    entity = result.get('entities', {}).get('Q42', {})
    label = entity.get('labels', {}).get('en', {}).get('value')
    desc = entity.get('descriptions', {}).get('en', {}).get('value')
    print(f"Label: {label}\nDescription: {desc}")

print("\nLooking up P31 (instance of):")
result = lookup_wikidata('P31', is_id=True)
if isinstance(result, dict):
    entity = result.get('entities', {}).get('P31', {})
    label = entity.get('labels', {}).get('en', {}).get('value')
    desc = entity.get('descriptions', {}).get('en', {}).get('value')
    print(f"Label: {label}\nDescription: {desc}")
```

This enhanced function:
1. Can search by text (like before)
2. Can look up directly by QID/PID
3. Returns full entity data including labels and descriptions

Should we run this and see how it works with both directions?

In [ ]:
def lookup_wikidata(id_or_text, search_type='item', is_id=False):
    """
    Search or lookup Wikidata entities/properties
    id_or_text: QID/PID (if is_id=True) or search text
    search_type: 'item' or 'property'
    is_id: True if looking up by ID, False to search by text
    """
    url = 'https://www.wikidata.org/w/api.php'
    
    if is_id:
        params = {
            'action': 'wbgetentities',
            'format': 'json',
            'languages': 'en',
            'ids': id_or_text
        }
    else:
        params = {
            'action': 'wbsearchentities',
            'format': 'json',
            'language': 'en',
            'search': id_or_text,
            'type': search_type
        }
    
    try:
        with httpx.Client() as client:
            r = client.get(url, params=params)
            r.raise_for_status()
            return r.json()
    except Exception as e: return f'Error: {e}'

# Test direct ID lookups
print("Looking up Q42 (Douglas Adams):")
result = lookup_wikidata('Q42', is_id=True)
if isinstance(result, dict):
    entity = result.get('entities', {}).get('Q42', {})
    label = entity.get('labels', {}).get('en', {}).get('value')
    desc = entity.get('descriptions', {}).get('en', {}).get('value')
    print(f"Label: {label}\nDescription: {desc}")

print("\nLooking up P31 (instance of):")
result = lookup_wikidata('P31', is_id=True)
if isinstance(result, dict):
    entity = result.get('entities', {}).get('P31', {})
    label = entity.get('labels', {}).get('en', {}).get('value')
    desc = entity.get('descriptions', {}).get('en', {}).get('value')
    print(f"Label: {label}\nDescription: {desc}")

Looking up Q42 (Douglas Adams):
Label: None
Description: English science fiction writer and humorist (1952–2001)

Looking up P31 (instance of):
Label: instance of
Description: that class of which this subject is a particular example and member; different from P279 (subclass of); for example: K2 is an instance of mountain; volcano is a subclass of mountain (and an instance of volcanic landform)


**AI Prompt**

OK I want to use claudette to make these functions into a tool. Let's do this in the SolveIt method so we can test each step. We may want to use the toolloop to understand what the agent is doing. Here is the prototype for toolloop:

@patch
@delegates(Chat.__call__)
def toolloop(self:Chat,
             pr, # Prompt to pass to Claude
             max_steps=10, # Maximum number of tool requests to loop through
             trace_func:Optional[callable]=None, # Function to trace tool use steps (e.g `print`)
             cont_func:Optional[callable]=noop, # Function that stops loop if returns False
             **kwargs):
    "Add prompt `pr` to dialog and get a response from Claude, automatically following up with `tool_use` messages"
    n_msgs = len(self.h)
    r = self(pr, **kwargs)
    for i in range(max_steps):
        if r.stop_reason!='tool_use': break
        if trace_func: trace_func(self.h[n_msgs:]); n_msgs = len(self.h)
        r = self(**kwargs)
        if not (cont_func or noop)(self.h[-2]): break
    if trace_func: trace_func(self.h[n_msgs:])
    return r

We’ll start by re-running our previous request - we shouldn’t have to manually pass back the tool_use message any more:

chat = Chat(model, tools=tools)
r = chat.toolloop('Can you tell me the email address for customer C1?')
r

**AI Response**

Let's build this step by step using Polya's method and Claudette's toolloop. First, we'll create our Wikidata tools in a format that Claudette can use:

```python
def wikidata_search(
    search_text: str,  # Text to search for
    search_type: str = 'item'  # Type of search: 'item' or 'property'
) -> dict:  # Returns search results with labels and descriptions
    """Search Wikidata for entities or properties by text"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': search_type
    }
    
    with httpx.Client() as client:
        r = client.get(url, params=params)
        r.raise_for_status()
        return r.json().get('search', [])[:3]  # Return top 3 matches

def wikidata_lookup(
    entity_id: str  # Wikidata ID (e.g., 'Q42' or 'P31')
) -> dict:  # Returns entity details with labels and descriptions
    """Look up a Wikidata entity or property by ID"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbgetentities',
        'format': 'json',
        'languages': 'en',
        'ids': entity_id
    }
    
    with httpx.Client() as client:
        r = client.get(url, params=params)
        r.raise_for_status()
        result = r.json()
        entity = result.get('entities', {}).get(entity_id, {})
        return {
            'label': entity.get('labels', {}).get('en', {}).get('value'),
            'description': entity.get('descriptions', {}).get('en', {}).get('value')
        }

# Create chat with our tools
chat = Chat(model, sp="You are a helpful assistant for querying Wikidata.", 
            tools=[wikidata_search, wikidata_lookup])

# Test with toolloop
r = chat.toolloop("Find information about Douglas Adams", trace_func=print)
```

This setup:
1. Creates two clean, documented tools for Claudette
2. Uses type hints and descriptions for better LLM understanding
3. Returns simplified results to reduce token usage
4. Uses toolloop to show us the process

Shall we run this and see how Claudette uses these tools to gather information?

In [ ]:
def wikidata_search(
    search_text: str,  # Text to search for
    search_type: str = 'item'  # Type of search: 'item' or 'property'
) -> dict:  # Returns search results with labels and descriptions
    """Search Wikidata for entities or properties by text"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbsearchentities',
        'format': 'json',
        'language': 'en',
        'search': search_text,
        'type': search_type
    }
    
    with httpx.Client() as client:
        r = client.get(url, params=params)
        r.raise_for_status()
        return r.json().get('search', [])[:3]  # Return top 3 matches

def wikidata_lookup(
    entity_id: str  # Wikidata ID (e.g., 'Q42' or 'P31')
) -> dict:  # Returns entity details with labels and descriptions
    """Look up a Wikidata entity or property by ID"""
    url = 'https://www.wikidata.org/w/api.php'
    params = {
        'action': 'wbgetentities',
        'format': 'json',
        'languages': 'en',
        'ids': entity_id
    }
    
    with httpx.Client() as client:
        r = client.get(url, params=params)
        r.raise_for_status()
        result = r.json()
        entity = result.get('entities', {}).get(entity_id, {})
        return {
            'label': entity.get('labels', {}).get('en', {}).get('value'),
            'description': entity.get('descriptions', {}).get('en', {}).get('value')
        }

# Lets select a model
model = models[1]
# Create chat with our tools
chat = Chat(model, sp="You are a helpful assistant for querying Wikidata.", 
            tools=[wikidata_search, wikidata_lookup])

# Test with toolloop
r = chat.toolloop("Find information about Douglas Adams", trace_func=print)

[{'role': 'user', 'content': 'Find information about Douglas Adams'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll help you search Wikidata for information about Douglas Adams.", type='text'), ToolUseBlock(id='toolu_012Skm2bqM4aSffYiWGpYPEN', input={'search_text': 'Douglas Adams'}, name='wikidata_search', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_012Skm2bqM4aSffYiWGpYPEN', 'content': "[{'id': 'Q28421831', 'title': 'Q28421831', 'pageid': 30117550, 'concepturi': 'http://www.wikidata.org/entity/Q28421831', 'repository': 'wikidata', 'url': '//www.wikidata.org/wiki/Q28421831', 'display': {'label': {'value': 'Douglas Adams', 'language': 'en'}, 'description': {'value': 'American environmental engineer', 'language': 'en'}}, 'label': 'Douglas Adams', 'description': 'American environmental engineer', 'match': {'type': 'label', 'language': 'en', 'text': 'Douglas Adams'}}, {'id': 'Q42', 'title': 'Q42', 'pageid': 138, 'c

## Role Prompting from Anthropic
[Giving Claude a role with a system prompt](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/system-prompts?utm_source=chatgpt.com#financial-analysis-with-role-prompting)

Prompt engineering

# Giving Claude a role with a system prompt

When using Claude, you can dramatically improve its performance by using the `system` parameter to give it a role. This technique, known as role prompting, is the most powerful way to use system prompts with Claude.

The right role can turn Claude from a general assistant into your virtual domain expert!

**System prompt tips** : Use the `system` parameter to set Claude’s role. Put everything else, like task-specific instructions, in the `user` turn instead.


Why use role prompting?

  * **Enhanced accuracy:** In complex scenarios like legal analysis or financial modeling, role prompting can significantly boost Claude’s performance.
  * **Tailored tone:** Whether you need a CFO’s brevity or a copywriter’s flair, role prompting adjusts Claude’s communication style.
  * **Improved focus:** By setting the role context, Claude stays more within the bounds of your task’s specific requirements.

How to give Claude a role

Use the `system` parameter in the Messages API to set Claude’s role:

```
import anthropic

client = anthropic.Anthropic()

response = client.messages.create(
    model="claude-3-5-sonnet-20241022",
    max_tokens=2048,
    system="You are a seasoned data scientist at a Fortune 500 company.", # <-- role prompt
    messages=[
        {"role": "user", "content": "Analyze this dataset for anomalies: <dataset>{{DATASET}}</dataset>"}
    ]
)

print(response.content)

```



**Role prompting tip** : Experiment with roles! A `data scientist` might see different insights than a `marketing strategist` for the same data. A `data scientist specializing in customer insight analysis for Fortune 500 companies` might yield different results still!

## Self Correction Chains

[Chain complex prompts for stronger performance](https://docs.anthropic.com/en/docs/build-with-claude/prompt-engineering/chain-prompts#advanced-self-correction-chains)

# Chain complex prompts for stronger performance

When working with complex tasks, Claude can sometimes drop the ball if you try to handle everything in a single prompt. Chain of thought (CoT) prompting is great, but what if your task has multiple distinct steps that each require in-depth thought?

Enter prompt chaining: breaking down complex tasks into smaller, manageable subtasks.


Why chain prompts?

  1. **Accuracy** : Each subtask gets Claude’s full attention, reducing errors.
  2. **Clarity** : Simpler subtasks mean clearer instructions and outputs.
  3. **Traceability** : Easily pinpoint and fix issues in your prompt chain.

When to chain prompts

Use prompt chaining for multi-step tasks like research synthesis, document analysis, or iterative content creation. When a task involves multiple transformations, citations, or instructions, chaining prevents Claude from dropping or mishandling steps.

**Remember:** Each link in the chain gets Claude’s full attention!

**Debugging tip** : If Claude misses a step or performs poorly, isolate that step in its own prompt. This lets you fine-tune problematic steps without redoing the entire task
​
How to chain prompts

  1. **Identify subtasks** : Break your task into distinct, sequential steps.
  2. **Structure with XML for clear handoffs** : Use XML tags to pass outputs between prompts.
  3. **Have a single-task goal** : Each subtask should have a single, clear objective.
  4. **Iterate** : Refine subtasks based on Claude’s performance.
  Advanced: Self-correction chains

You can chain prompts to have Claude review its own work! This catches errors and refines outputs, especially for high-stakes tasks.

#### Prompt 1

Role| Content  
---|---  
User| Summarize this medical research paper.  
  
<paper>{{RESEARCH_PAPER}}</paper>  
  
Focus on methodology, findings, and clinical implications.  
Assistant  
{{SUMMARY}}| Methodology: Double-blind RCT with 500 Type 2 diabetes patients. Group A (n=250) received novel GLP-1/GIP dual agonist; Group B (n=250) received standard GLP-1 agonist. 24-week trial, primary endpoint: HbA1c reduction.  
  
Findings:  
\- Group A: 1.8% mean HbA1c reduction  
\- Group B: 1.1% mean HbA1c reduction  
\- 68% of Group A vs. 52% of Group B achieved HbA1c <7%  
\- Group A: 5.2kg mean weight loss  
\- Group B: 3.1kg mean weight loss  
  
Safety: Similar adverse event profiles, no serious events attributed to treatment.  
  
Clinical Implications: Dual agonist shows superior glycemic control and weight loss compared to standard therapy. Could be a game-changer, especially for obese diabetics.  
  
#### Prompt 2

Role| Content  
---|---  
User| Your task is to provide feedback on a research paper summary. Here is a summary of a medical research paper:  
<summary>  
{{SUMMARY}}  
</summary>  
  
Here is the research paper:  
<paper>  
{{RESEARCH_PAPER}}  
</paper>  
  
Review this summary for accuracy, clarity, and completeness on a graded A-F scale.  
Assistant  
{{FEEDBACK}}| 1\. Accuracy: A  
\- Correctly reported key stats: HbA1c reductions, weight loss.  
\- Accurately noted safety profile and lack of serious adverse events.  
\- Assertion of “game-changer” is justified by significantly better outcomes.  
  
2\. Clarity: A-  
\- Concise and well-structured with clear sections.  
\- Good use of comparative language (e.g., “up 3% from Q1”) for context.  
\- Minor issue: Acronyms like “RCT,” “GLP-1,” and “GIP” not defined. Consider spelling out first use.  
  
3\. Completeness: B+  
\- Covers all requested areas: methodology, findings, clinical implications.  
\- Missing: Baseline characteristics of groups (were they well-matched?).  
\- Missing: Any secondary endpoints (e.g., blood pressure, lipids).  
\- Consider: Long-term implications or need for further research.  
  
Overall, the summary is accurate and clear, providing key insights for a quick grasp of the study. To elevate to A+, define acronyms, mention group baseline stats, note any secondary outcomes, and briefly touch on the need for long-term studies.  
  
#### Prompt 3

Role| Content  
---|---  
User| Your task is to improve a paper summary given feedback. Here is the first draft of a medical research paper:  
<summary>  
{{SUMMARY}}  
</summary>  
  
Here is the research paper:  
<paper>  
{{RESEARCH_PAPER}}  
</paper>  
  
Here is the feedback:  
<feedback>  
{{FEEDBACK}}  
</feedback>  
  
Update the summary based on the feedback.  
Assistant| Methodology: Double-blind randomized controlled trial (RCT) with 500 Type 2 diabetes patients. Group A (n=250) received a novel glucagon-like peptide-1/glucose-dependent insulinotropic polypeptide (GLP-1/GIP) dual agonist; Group B (n=250) received a standard GLP-1 agonist. The 24-week trial’s primary endpoint was HbA1c reduction. Groups were well-matched for age, BMI, and baseline HbA1c.  
  
Findings:  
\- Group A: 1.8% mean HbA1c reduction  
\- Group B: 1.1% mean HbA1c reduction  
\- 68% of Group A vs. 52% of Group B achieved HbA1c <7%  
\- Group A: 5.2kg mean weight loss  
\- Group B: 3.1kg mean weight loss  
  
Secondary endpoints showed improvements in blood pressure and lipid profiles for both groups, with Group A showing slightly better results.  
  
Safety: Similar adverse event profiles, with no serious events attributed to either treatment.  
  
Clinical Implications: The dual GLP-1/GIP agonist demonstrates superior glycemic control and weight loss compared to standard GLP-1 therapy. These results suggest it could be a significant advancement, particularly for obese diabetics. However, longer-term studies are needed to confirm sustained efficacy and safety, as well as to investigate potential benefits on cardiovascular outcomes.

**AI Prompt**

Ok. I want to experiment with extending our example with prompt chaining to correct potential SPARQL query errors. The most common wikidata query errors are hallucination of QIDs and PIDs, so I want to focus on that first.

**AI Response**

Let's create a self-correcting chain for Wikidata queries that focuses on QID/PID verification. We'll modify our setup to include multiple steps:

```python
def create_wikidata_assistant():
    # Role prompt focusing on accuracy and verification
    sp = """You are a Wikidata query specialist who is extremely careful about entity and property IDs.
    Before using any QID or PID, you MUST verify it exists using the lookup tools.
    Always check search results before using any ID."""
    
    chat = Chat(model, sp=sp, tools=[wikidata_search, wikidata_lookup])
    return chat

def verify_ids(query_text):
    """Chain of prompts to verify and correct Wikidata IDs"""
    chat = create_wikidata_assistant()
    
    # Step 1: Initial search and ID verification
    prompt1 = f"""Find and verify Wikidata IDs for this query: "{query_text}"
    First search for relevant entities and properties, then verify each ID."""
    
    r = chat.toolloop(prompt1, trace_func=print)
    
    # Step 2: Self-correction check
    prompt2 = f"""Given these verified IDs, check if any might be incorrect or hallucinated.
    <previous_response>
    {r.content}
    </previous_response>
    
    For each ID mentioned, explicitly state whether it was verified successfully."""
    
    return chat.toolloop(prompt2, trace_func=print)

# Test the chain
test_query = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = verify_ids(test_query)
```

This setup:
1. Creates a specialized role for ID verification
2. Uses toolloop to show the verification process
3. Implements a two-step chain:
   - First step finds and verifies IDs
   - Second step double-checks for hallucinations
4. Uses our existing tools but with more careful verification

Should we run this and see how it handles ID verification?

In [ ]:
def create_wikidata_assistant():
    # Role prompt focusing on accuracy and verification
    sp = """You are a Wikidata query specialist who is extremely careful about entity and property IDs.
    Before using any QID or PID, you MUST verify it exists using the lookup tools.
    Always check search results before using any ID."""
    
    chat = Chat(model, sp=sp, tools=[wikidata_search, wikidata_lookup])
    return chat

def verify_ids(query_text):
    """Chain of prompts to verify and correct Wikidata IDs"""
    chat = create_wikidata_assistant()
    
    # Step 1: Initial search and ID verification
    prompt1 = f"""Find and verify Wikidata IDs for this query: "{query_text}"
    First search for relevant entities and properties, then verify each ID."""
    
    r = chat.toolloop(prompt1, trace_func=print)
    
    # Step 2: Self-correction check
    prompt2 = f"""Given these verified IDs, check if any might be incorrect or hallucinated.
    <previous_response>
    {r.content}
    </previous_response>
    
    For each ID mentioned, explicitly state whether it was verified successfully."""
    
    return chat.toolloop(prompt2, trace_func=print)

# Test the chain
test_query = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = verify_ids(test_query)

[{'role': 'user', 'content': 'Find and verify Wikidata IDs for this query: "Who wrote The Hitchhiker\'s Guide to the Galaxy?"\n    First search for relevant entities and properties, then verify each ID.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='I\'ll help search for and verify the relevant Wikidata IDs for this query. We need to find:\n1. "The Hitchhiker\'s Guide to the Galaxy" (the work)\n2. A property for "wrote" or "author"\n\nLet\'s search for these one by one:\n\nFirst, let\'s search for the book:', type='text'), ToolUseBlock(id='toolu_01394qETmXr2eGUKyPfecDVp', input={'search_text': "The Hitchhiker's Guide to the Galaxy", 'search_type': 'item'}, name='wikidata_search', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01394qETmXr2eGUKyPfecDVp', 'content': '[{\'id\': \'Q3107329\', \'title\': \'Q3107329\', \'pageid\': 2969355, \'concepturi\': \'http://www.wikidata.org/entity/Q3107329\', \'repository\': \'wikida

**AI Prompt**

Ok. This is good. But we want to have an LLM interpert the results back in natural language and cite its source.

**AI Response**

Let's add a third step to our chain that interprets the results in natural language with citations. We'll modify our verification function:

```python
def verify_and_interpret_wikidata(query_text):
    """Chain of prompts to verify IDs and interpret results with citations"""
    chat = create_wikidata_assistant()
    
    # Step 1: Initial search and ID verification
    prompt1 = f"""Find and verify Wikidata IDs for this query: "{query_text}"
    First search for relevant entities and properties, then verify each ID."""
    
    r1 = chat.toolloop(prompt1, trace_func=print)
    
    # Step 2: Self-correction check
    prompt2 = f"""Given these verified IDs, check if any might be incorrect or hallucinated.
    <previous_response>
    {r1.content}
    </previous_response>
    
    For each ID mentioned, explicitly state whether it was verified successfully."""
    
    r2 = chat.toolloop(prompt2, trace_func=print)
    
    # Step 3: Natural language interpretation with citations
    prompt3 = f"""Based on the verified Wikidata information, provide a natural language response to the original query: "{query_text}"
    
    <verification_results>
    {r2.content}
    </verification_results>
    
    Please:
    1. Answer in natural language
    2. Include Wikidata entity IDs in parentheses where relevant
    3. Cite specific verification steps that support your answer"""
    
    return chat.toolloop(prompt3, trace_func=print)

# Test the enhanced chain
test_query = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = verify_and_interpret_wikidata(test_query)
```

This enhancement:
1. Keeps our careful verification steps
2. Adds natural language interpretation
3. Requires citations to verification steps
4. Makes the results more user-friendly while maintaining traceability

Should we run this and see how it handles the full chain of verification and interpretation?

In [ ]:
def verify_and_interpret_wikidata(query_text):
    """Chain of prompts to verify IDs and interpret results with citations"""
    chat = create_wikidata_assistant()
    
    # Step 1: Initial search and ID verification
    prompt1 = f"""Find and verify Wikidata IDs for this query: "{query_text}"
    First search for relevant entities and properties, then verify each ID."""
    
    r1 = chat.toolloop(prompt1, trace_func=print)
    
    # Step 2: Self-correction check
    prompt2 = f"""Given these verified IDs, check if any might be incorrect or hallucinated.
    <previous_response>
    {r1.content}
    </previous_response>
    
    For each ID mentioned, explicitly state whether it was verified successfully."""
    
    r2 = chat.toolloop(prompt2, trace_func=print)
    
    # Step 3: Natural language interpretation with citations
    prompt3 = f"""Based on the verified Wikidata information, provide a natural language response to the original query: "{query_text}"
    
    <verification_results>
    {r2.content}
    </verification_results>
    
    Please:
    1. Answer in natural language
    2. Include Wikidata entity IDs in parentheses where relevant
    3. Cite specific verification steps that support your answer"""
    
    return chat.toolloop(prompt3, trace_func=print)

# Test the enhanced chain
test_query = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = verify_and_interpret_wikidata(test_query)

[{'role': 'user', 'content': 'Find and verify Wikidata IDs for this query: "Who wrote The Hitchhiker\'s Guide to the Galaxy?"\n    First search for relevant entities and properties, then verify each ID.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='I\'ll help search for and verify the relevant Wikidata IDs for this query. We need to find:\n1. "The Hitchhiker\'s Guide to the Galaxy" (the work)\n2. A property for "wrote" or "author"\n\nLet\'s search for these one by one:\n\nFirst, let\'s search for the book:', type='text'), ToolUseBlock(id='toolu_01Y752A5thYcLuMecdqd1NR5', input={'search_text': "The Hitchhiker's Guide to the Galaxy", 'search_type': 'item'}, name='wikidata_search', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01Y752A5thYcLuMecdqd1NR5', 'content': '[{\'id\': \'Q3107329\', \'title\': \'Q3107329\', \'pageid\': 2969355, \'concepturi\': \'http://www.wikidata.org/entity/Q3107329\', \'repository\': \'wikida

**AI Prompt**

Ok. Let's add SPARQL to the mix. We want an sparql query constructor agent that construct a SPARQL query based on a natural language question. A corrector agent that makes sure the QIDs and PIDs are correct in the query and any other agents as necessary from the previous chain.

**AI Response**

Let's create a multi-agent chain that constructs, verifies, and executes SPARQL queries. Here's the proposed structure:

```python
def create_sparql_constructor():
    sp = """You are a SPARQL query construction specialist. 
    Your role is to convert natural language questions into Wikidata SPARQL queries.
    Always use PREFIX wd: <http://www.wikidata.org/entity/>
    Always use PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    Do not include actual QIDs or PIDs in the query template - use placeholders like {BOOK_ID} and {AUTHOR_PROP}."""
    
    return Chat(model, sp=sp)

def create_query_corrector():
    sp = """You are a Wikidata query correction specialist.
    Your role is to verify and correct QIDs and PIDs in SPARQL queries.
    Always verify each ID before replacing placeholders in the query."""
    
    return Chat(model, sp=sp, tools=[wikidata_search, wikidata_lookup])

def sparql_pipeline(question: str):
    """Multi-agent pipeline for SPARQL query construction and verification"""
    
    # Step 1: Construct SPARQL template
    constructor = create_sparql_constructor()
    prompt1 = f"""Convert this question to a SPARQL query template: "{question}"
    Use placeholders for QIDs and PIDs."""
    
    query_template = constructor(prompt1)
    
    # Step 2: Identify and verify entities/properties
    verifier = verify_and_interpret_wikidata(question)
    
    # Step 3: Correct query with verified IDs
    corrector = create_query_corrector()
    prompt3 = f"""Replace placeholders in this SPARQL template with verified IDs:
    
    <template>
    {query_template.content}
    </template>
    
    <verified_ids>
    {verifier.content}
    </verified_ids>"""
    
    final_query = corrector.toolloop(prompt3, trace_func=print)
    
    return final_query

# Test the pipeline
test_question = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = sparql_pipeline(test_question)
```

This setup:
1. Creates a SPARQL constructor that makes query templates
2. Reuses our ID verification chain
3. Creates a corrector that combines verified IDs with the template
4. Uses toolloop to show the process at each step

The flow is:
1. Natural language → SPARQL template with placeholders
2. Parallel ID verification process
3. Template + verified IDs → final SPARQL query

Should we run this and see how the agents work together?

In [ ]:
def create_sparql_constructor():
    sp = """You are a SPARQL query construction specialist. 
    Your role is to convert natural language questions into Wikidata SPARQL queries.
    Always use PREFIX wd: <http://www.wikidata.org/entity/>
    Always use PREFIX wdt: <http://www.wikidata.org/prop/direct/>
    Do not include actual QIDs or PIDs in the query template - use placeholders like {BOOK_ID} and {AUTHOR_PROP}."""
    
    return Chat(model, sp=sp)

def create_query_corrector():
    sp = """You are a Wikidata query correction specialist.
    Your role is to verify and correct QIDs and PIDs in SPARQL queries.
    Always verify each ID before replacing placeholders in the query."""
    
    return Chat(model, sp=sp, tools=[wikidata_search, wikidata_lookup])

def sparql_pipeline(question: str):
    """Multi-agent pipeline for SPARQL query construction and verification"""
    
    # Step 1: Construct SPARQL template
    constructor = create_sparql_constructor()
    prompt1 = f"""Convert this question to a SPARQL query template: "{question}"
    Use placeholders for QIDs and PIDs."""
    
    query_template = constructor(prompt1)
    
    # Step 2: Identify and verify entities/properties
    verifier = verify_and_interpret_wikidata(question)
    
    # Step 3: Correct query with verified IDs
    corrector = create_query_corrector()
    prompt3 = f"""Replace placeholders in this SPARQL template with verified IDs:
    
    <template>
    {query_template.content}
    </template>
    
    <verified_ids>
    {verifier.content}
    </verified_ids>"""
    
    final_query = corrector.toolloop(prompt3, trace_func=print)
    
    return final_query

# Test the pipeline
test_question = "Who wrote The Hitchhiker's Guide to the Galaxy?"
result = sparql_pipeline(test_question)

[{'role': 'user', 'content': 'Find and verify Wikidata IDs for this query: "Who wrote The Hitchhiker\'s Guide to the Galaxy?"\n    First search for relevant entities and properties, then verify each ID.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='I\'ll help search for and verify the relevant Wikidata IDs for this query. We need to find:\n1. "The Hitchhiker\'s Guide to the Galaxy" (the work)\n2. A property for "wrote" or "author"\n\nLet\'s search for these one by one:\n\nFirst, let\'s search for the book:', type='text'), ToolUseBlock(id='toolu_01NLS6Rm1Xj8FJapwmAtTmhR', input={'search_text': "The Hitchhiker's Guide to the Galaxy", 'search_type': 'item'}, name='wikidata_search', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01NLS6Rm1Xj8FJapwmAtTmhR', 'content': '[{\'id\': \'Q3107329\', \'title\': \'Q3107329\', \'pageid\': 2969355, \'concepturi\': \'http://www.wikidata.org/entity/Q3107329\', \'repository\': \'wikida

In [ ]:
# Test the pipeline
test_question = "Who wrote Hitchhikers Guide to the Galaxy and what year was it written?"
result = sparql_pipeline(test_question)

[{'role': 'user', 'content': 'Find and verify Wikidata IDs for this query: "Who wrote Hitchhikers Guide to the Galaxy and what year was it written?"\n    First search for relevant entities and properties, then verify each ID.'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='I\'ll help search for and verify the relevant Wikidata IDs. Let\'s break this down:\n\n1. First, let\'s search for "Hitchhiker\'s Guide to the Galaxy":', type='text'), ToolUseBlock(id='toolu_01QQBuG5PFyzT3itXX1ZVg6Q', input={'search_text': "Hitchhiker's Guide to the Galaxy"}, name='wikidata_search', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01QQBuG5PFyzT3itXX1ZVg6Q', 'content': '[{\'id\': \'Q3107329\', \'title\': \'Q3107329\', \'pageid\': 2969355, \'concepturi\': \'http://www.wikidata.org/entity/Q3107329\', \'repository\': \'wikidata\', \'url\': \'//www.wikidata.org/wiki/Q3107329\', \'display\': {\'label\': {\'value\': "The Hitchhiker\'s Guide 